In [1]:
import sys
import os
BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(BASE_PATH)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from src.data_loader import get_dataloaders

BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_PATH = os.path.join(BASE_PATH, "data", "raw")

train_loader, val_loader, num_brands = get_dataloaders(DATASET_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Dataset size: 4000
Class distribution:
 label
0    2000
1    2000
Name: count, dtype: int64
Brands: 10
Using device: cuda


In [2]:
# Load ResNet50 model
# model = models.resnet50(pretrained=True)

In [3]:
# Modify final layer for binary classification

class BrandAwareResNet(nn.Module):
    def __init__(self, base_model, num_brands):
        super().__init__()
        self.base = base_model
        self.base.fc = nn.Identity()  # remove original FC

        self.brand_embedding = nn.Embedding(num_brands, 16)

        self.classifier = nn.Sequential(
            nn.Linear(2048 + 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x, brand):
        features = self.base(x)
        brand_feat = self.brand_embedding(brand)

        combined = torch.cat([features, brand_feat], dim=1)
        out = self.classifier(combined)

        return out

# model = model.to(device)

In [4]:
# Load base model
base_model = models.resnet50(pretrained=True)

# Create final model
model = BrandAwareResNet(base_model, num_brands).to(device)

print("Model initialized with", num_brands, "brands")

d:\VIT Personal\TrustFilterAI\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\VIT Personal\TrustFilterAI\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model initialized with 10 brands


In [5]:
# Freeze early layers, fine-tune later layers and classifier
for param in model.base.parameters():
    param.requires_grad = False

for param in model.base.layer3.parameters():
    param.requires_grad = True

for param in model.base.layer4.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.brand_embedding.parameters():
    param.requires_grad = True

In [6]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.00005, weight_decay=1e-4)

In [7]:
# Training loop
def train_model(model, train_loader, val_loader, epochs=15):
    best_val_acc = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels, brands in train_loader:
            images, labels, brands = images.to(device), labels.to(device), brands.to(device)

            optimizer.zero_grad()

            outputs = model(images, brands)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0

        with torch.no_grad():
            for images, labels, brands in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                brands = brands.to(device)

                outputs = model(images, brands)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "../models/best_model.pth")

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} "
              f"Val Acc: {val_acc:.4f}")

    print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

In [8]:
# Train the model
train_model(model, train_loader, val_loader, epochs=10)

Epoch [1/10] Train Loss: 0.3478 Train Acc: 0.8413 Val Loss: 0.1717 Val Acc: 0.9450
Epoch [2/10] Train Loss: 0.1501 Train Acc: 0.9469 Val Loss: 0.1361 Val Acc: 0.9563
Epoch [3/10] Train Loss: 0.0973 Train Acc: 0.9631 Val Loss: 0.1329 Val Acc: 0.9475
Epoch [4/10] Train Loss: 0.0624 Train Acc: 0.9781 Val Loss: 0.1329 Val Acc: 0.9487
Epoch [5/10] Train Loss: 0.0596 Train Acc: 0.9781 Val Loss: 0.1560 Val Acc: 0.9375
Epoch [6/10] Train Loss: 0.0421 Train Acc: 0.9875 Val Loss: 0.1231 Val Acc: 0.9587
Epoch [7/10] Train Loss: 0.0358 Train Acc: 0.9888 Val Loss: 0.1186 Val Acc: 0.9650
Epoch [8/10] Train Loss: 0.0335 Train Acc: 0.9875 Val Loss: 0.1218 Val Acc: 0.9563
Epoch [9/10] Train Loss: 0.0275 Train Acc: 0.9906 Val Loss: 0.1528 Val Acc: 0.9525
Epoch [10/10] Train Loss: 0.0246 Train Acc: 0.9903 Val Loss: 0.1534 Val Acc: 0.9513

Best Validation Accuracy: 0.9650


In [9]:
torch.save(model.state_dict(), "../models/resnet50_counterfeit.pth")
print("Model saved")

Model saved
